# 02 — Chat Models with Gemini

In notebook 01 we called the model with a plain string. That was a convenience. The
real interface to a modern chat model is a **list of messages**, each with a role.
Understanding messages unlocks system instructions, multi-turn conversations, and
(later) tool calling.

In this notebook:

1. The message types and their roles.
2. Why the **system message** matters.
3. Holding a multi-turn conversation by hand.
4. The parameters that shape responses (`temperature`, `max_output_tokens`, ...).
5. **Streaming** and **batching**.
6. Reading **token usage** to understand cost.
7. A taste of **multimodal** input (sending an image).

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
print("Model ready.")

Model ready.


## 1. Messages and roles

A chat model sees a conversation as a sequence of messages. The main roles:

| Class | Role | Purpose |
|-------|------|---------|
| `SystemMessage` | system | Sets the model's behavior, persona, and rules. Comes first. |
| `HumanMessage` | user | What the user says. |
| `AIMessage` | assistant | What the model said previously (you include these to give it memory of the turn). |

`.invoke()` accepts a **list** of these. Let's send a system message plus a human
message.

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a terse assistant. Answer in one short sentence."),
    HumanMessage(content="Why is the sky blue?"),
]

response = model.invoke(messages)
print(response.content)

Atmospheric gases scatter blue light more effectively than other colors.


### The tuple shorthand

Typing the message classes is explicit but verbose. LangChain also accepts a list of
`(role, text)` tuples, which is handy for quick work. These two calls are equivalent.

In [3]:
response = model.invoke([
    ("system", "You are a terse assistant. Answer in one short sentence."),
    ("human", "Why is the sky blue?"),
])
print(response.content)

The sky is blue because blue light is scattered more by Earth's atmosphere than other colors.


## 2. The system message is your control knob

The system message is where you set tone, format, persona, and rules. Changing it
changes everything downstream without touching the user's question. Run the cell to see
the same question answered by three different "personalities".

In [4]:
question = "How do I reverse a list in Python?"

personas = {
    "Terse expert":     "You answer in one line of code, no prose.",
    "Patient teacher":  "You explain step by step for a beginner, with a short example.",
    "Pirate":           "You answer correctly but speak like a pirate.",
}

for name, system in personas.items():
    out = model.invoke([("system", system), ("human", question)])
    print(f"### {name}\n{out.content}\n")

### Terse expert
`list[::-1]`

### Patient teacher
Reversing a list in Python is a common task, and there are a few straightforward ways to do it. For a beginner, it's good to understand two main approaches: one that changes the original list, and one that creates a new, reversed list.

Let's break it down step by step!

---

### Method 1: Using `list.reverse()` (Modifies the original list)

This method is a built-in "method" that belongs to list objects. It changes the list *itself* directly.

**Step 1: Create your list.**
Let's start with a simple list of numbers.

```python
my_numbers = [1, 2, 3, 4, 5]
print("Original list:", my_numbers)
# Output: Original list: [1, 2, 3, 4, 5]
```

**Step 2: Call the `.reverse()` method on your list.**
You don't need to assign the result to a new variable because it changes the list "in-place."

```python
my_numbers.reverse()
```

**Step 3: Print your list again to see the change.**

```python
print("Reversed list (in-place):", my_numbers)
# Output

## 3. A multi-turn conversation, by hand

The model is **stateless** — it does not remember previous calls. To continue a
conversation you resend the whole history, including the model's own previous
`AIMessage`. Here we follow up on an earlier answer and the "it" is understood only
because the prior turn is in the list.

In [ ]:
conversation = [
    SystemMessage(content="You are a helpful Python tutor."),
    HumanMessage(content="What is a list comprehension?"),
]

# First turn
first = model.invoke(conversation)
print("Assistant:", first.content[:200], "...\n")

# Append the assistant's reply, then ask a follow-up that depends on it.
conversation.append(first)  # an AIMessage
conversation.append(HumanMessage(content="Now show me one that filters even numbers."))

second = model.invoke(conversation)
print("Assistant:", second.content)

Manually appending messages works, but it gets tedious and you must manage the growing
list yourself. Notebook 06 introduces tools that automate this. For now, the key idea:
**memory = resending history**.

## 4. Parameters that shape the output

You can set these on the constructor (applies to every call) or per call. The most useful:

- **`temperature`** (0–~1): randomness. Low = focused/repeatable, high = creative/varied.
- **`max_output_tokens`**: a hard cap on response length.
- **`top_p`**: nucleus sampling; another way to control diversity. Usually leave at default.

Let's contrast temperatures on a creative task.

In [ ]:
creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0)
focused  = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

prompt = "Invent a name for a coffee shop run by programmers. Reply with just the name."

print("temperature=0.0:")
print(" ", focused.invoke(prompt).content)
print(" ", focused.invoke(prompt).content)   # repeatable

print("\ntemperature=1.0:")
print(" ", creative.invoke(prompt).content)
print(" ", creative.invoke(prompt).content)   # varied

You can also cap length. Setting `max_output_tokens` low forces brevity (and can cut
the response off mid-sentence — that's expected).

In [ ]:
short = ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_output_tokens=40)
print(short.invoke("Explain how the internet works.").content)

> **Gotcha.** `max_output_tokens` limits the *output*, not your input. If a response
> looks truncated, this is usually why. Token counts are not word counts — a token is
> roughly 3–4 characters of English.

## 5. Streaming

`.invoke()` waits for the entire response. For anything user-facing, **streaming** feels
far better: you print tokens as they arrive. `.stream()` returns an iterator of message
chunks; concatenate their `.content`.

In [2]:
print("Streaming response:\n")
full = ""
for chunk in model.stream("List 5 uses of LangChain, one per line."):
    print(chunk.content, end="", flush=True)
    full += chunk.content

print("\n\n(collected", len(full), "characters)")

Streaming response:

Here are 5 uses of LangChain:

1.  Building conversational AI agents (chatbots) with memory and tool access.
2.  Implementing Retrieval Augmented Generation (RAG) for Q&A over custom data sources.
3.  Creating autonomous agents that can use multiple tools to complete complex, multi-step tasks.
4.  Developing multi-step data processing pipelines (e.g., summarization, extraction, transformation).
5.  Integrating LLMs with external APIs, databases, and other software systems.

(collected 477 characters)


## 6. Batching

When you have many independent inputs, `.batch()` runs them concurrently and returns the
results in order — much faster than a Python `for` loop of `.invoke()` calls.

In [ ]:
cities = ["Tokyo", "Cairo", "Oslo"]
prompts = [f"Give one interesting fact about {c}. One sentence." for c in cities]

results = model.batch(prompts)
for city, res in zip(cities, results):
    print(f"{city}: {res.content}")

> **Async note.** Every method has an `a`-prefixed async twin: `await model.ainvoke(...)`,
> `model.astream(...)`, `await model.abatch(...)`. Use them inside `async def` functions
> (web servers, for example). The synchronous versions are fine for notebooks.

## 7. Token usage and cost

Providers bill by tokens. Every `AIMessage` carries `usage_metadata` with the counts, so
you can measure and budget. This is the number to watch as your prompts grow (especially
once you start stuffing documents into them for RAG).

In [ ]:
resp = model.invoke("Write a haiku about debugging.")
print(resp.content)
print("\nToken usage:", resp.usage_metadata)

## 8. A taste of multimodal input

Gemini models are multimodal — a `HumanMessage` can contain both text and an image. You
pass `content` as a list of parts. Here we fetch a small public image and ask the model
to describe it.

(If you're offline, skip this cell — the rest of the course is text-only.)

In [7]:
import base64, httpx
from langchain_core.messages import HumanMessage

# A small public domain image.
image_url = "https://avatars.githubusercontent.com/u/96572607?v=4"
image_bytes = httpx.get(image_url).content
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

message = HumanMessage(content=[
    {"type": "text", "text": "Describe this image in one sentence."},
    {"type": "image_url", "image_url": {
            "url": f"data:image/jpeg;base64,{image_b64}"
        }
    },
])

print(model.invoke([message]).content)

A man with a beard and glasses, wearing a maroon hoodie and a dark jacket, smiles slightly while looking directly at the camera against a purple to blue gradient background.


## Your turn

Five exercises modelled on tasks you'd actually ship — a tone rewriter, a streamed
announcement, a bulk translator, a multi-turn SQL helper, and an accessibility alt-text
generator. Attempt each in a blank cell before opening its solution.

**Exercise 1 — Tone rewriter.** Write `rewrite(note: str, audience: str) -> str` that uses a
**system message** to turn a blunt internal note into a message for a given audience while
keeping the facts. Rewrite `"tell the customer no refund, policy is 14 days, they're at 20"`
for "a disappointed customer — polite, empathetic email".

*Sample run (illustrative):*

```text
Hi there,

Thank you for reaching out, and I'm sorry to hear you'd like to return your order. After
reviewing the details, I see the purchase was made 20 days ago, which falls outside our
14-day return window, so I'm unable to process a refund in this case.

I completely understand this is disappointing, and I'd be glad to look into other options
that might help. Please let me know.

Warm regards,
Support Team
```

<details><summary>Show solution</summary>

```python
def rewrite(note: str, audience: str) -> str:
    system = f"Rewrite rough internal notes into a message for: {audience}. Keep the facts; fix the tone."
    return model.invoke([("system", system), ("human", note)]).content

print(rewrite(
    "tell the customer no refund, policy is 14 days, they're at 20",
    "a disappointed customer — polite, empathetic email",
))
```

The system message is the knob: change the audience and the whole reply changes.

</details>


**Exercise 2 — Streamed, length-capped announcement.** Make a model with a low
`max_output_tokens` and **stream** a tweet-length launch announcement so it appears word by
word, like a live UI.

*Sample run (prints progressively, then stops at the token cap):*

```text
🔒 Your notes, your business. Meet Jotty — the privacy-first note app where everything stays
encrypted on your device. No tracking, no ads, just your ideas. #privacy
```

<details><summary>Show solution</summary>

```python
tweeter = ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_output_tokens=80, temperature=0.8)
for chunk in tweeter.stream("Write a punchy launch tweet for a privacy-first note-taking app."):
    print(chunk.content, end="", flush=True)
print()
```

`max_output_tokens` caps the length; `.stream()` makes it feel responsive.

</details>

**Exercise 3 — Localize the UI in one batch.** You have several interface strings to
translate to French. Use `.batch()` to translate them all concurrently, then pair each
English string with its translation.

*Sample run (illustrative):*

```text
Save changes       -> Enregistrer les modifications
Delete account     -> Supprimer le compte
Your cart is empty -> Votre panier est vide
Sign in            -> Se connecter
```

<details><summary>Show solution</summary>

```python
ui = ["Save changes", "Delete account", "Your cart is empty", "Sign in"]
batches = [
    [("system", "Translate to French. Return only the translation."), ("human", s)]
    for s in ui
]
for en, res in zip(ui, model.batch(batches)):
    print(f"{en:18} -> {res.content}")
```

`.batch()` fires all four off at once, so the wait is roughly one call, not four.

</details>

**Exercise 4 — Multi-turn SQL helper.** Hold a two-turn conversation. First ask for SQL that
finds the top 5 customers by total spend; then follow up with "now only count orders from the
last 30 days" and confirm the second answer builds on the first.

*Sample run (illustrative):*

```text
Q1:
 SELECT customer_id, SUM(amount) AS total_spend
 FROM orders
 GROUP BY customer_id
 ORDER BY total_spend DESC
 LIMIT 5;

Q2:
 SELECT customer_id, SUM(amount) AS total_spend
 FROM orders
 WHERE created_at >= NOW() - INTERVAL '30 days'
 GROUP BY customer_id
 ORDER BY total_spend DESC
 LIMIT 5;
```

<details><summary>Show solution</summary>

```python
convo = [SystemMessage(content="You are a senior data engineer. Return SQL only, no prose.")]
convo.append(HumanMessage(content="Table orders(customer_id, amount, created_at). Top 5 customers by total spend."))
q1 = model.invoke(convo); print("Q1:\n", q1.content, "\n")

convo.append(q1)  # the AIMessage carries the context forward
convo.append(HumanMessage(content="Now restrict it to orders from the last 30 days."))
print("Q2:\n", model.invoke(convo).content)
```

The model only understands the follow-up because the first query is back in the list.

</details>


**Exercise 5 — Alt-text generator (multimodal, optional).** Send an image and ask for a
one-sentence alt-text suitable for a screen reader. (Skip if you're offline.)

*Sample run (illustrative — depends on the image):*

```text
A close-up of a tabby cat with green eyes looking directly at the camera against a plain
background.
```

<details><summary>Show solution</summary>

```python
import base64, httpx
from langchain_core.messages import HumanMessage

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Orange_tabby_cat_sitting_on_fallen_leaves-Hisashi-01A.jpg/250px-Orange_tabby_cat_sitting_on_fallen_leaves-Hisashi-01A.jpg"
b64 = base64.b64encode(httpx.get(url).content).decode()
msg = HumanMessage(content=[
    {"type": "text", "text": "Write concise alt-text (one sentence) for this image, for a screen reader."},
    {"type": "image_url", "image_url": f"data:image/jpeg;base64,{b64}"},
])
print(model.invoke([msg]).content)
```

A genuine accessibility use case: generating alt-text for images at scale.

</details>

## Summary

- Chat models work on a **list of messages**: `SystemMessage`, `HumanMessage`,
  `AIMessage` (or `(role, text)` tuples).
- The **system message** controls behavior and format.
- Models are **stateless**; multi-turn conversations work by **resending history**.
- `temperature` controls randomness; `max_output_tokens` caps length.
- Use `.stream()` for responsive UIs, `.batch()` for many independent inputs, and the
  `a`-prefixed methods for async.
- `usage_metadata` reports token counts — your cost signal.

**Next:** [03 — Prompt Templates](03_Prompt_Templates.ipynb). Hard-coding prompt strings
doesn't scale; templates make prompts reusable and parameterized.